# Fraud Rules Evaluation - ClickHouse Direct Queries

This notebook executes fraud detection rules directly in ClickHouse and evaluates their performance.
Each rule is executed separately and results are stored in a pandas DataFrame for analysis.

## 1. Setup and Imports

In [1]:
import pandas as pd
from clickhouse_driver import Client
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. ClickHouse Connection Configuration

In [2]:
# ClickHouse configuration
CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

# Create ClickHouse client
client = Client(**CLICKHOUSE_CONFIG)

# Test connection
result = client.execute('SELECT version()')
print(f"✅ Connected to ClickHouse version: {result[0][0]}")

✅ Connected to ClickHouse version: 25.10.1.3796


## 3. Define Date Range and Parameters

In [3]:
# Date range for analysis
START_DATE = '2025-07-01'
END_DATE = '2025-07-30'

print(f"📅 Analysis Period: {START_DATE} to {END_DATE}")
print(f"📊 Table: public.stixor_fraud_features_distributed")
print(f"🔍 Filter: Customer Account only")

📅 Analysis Period: 2025-07-01 to 2025-07-30
📊 Table: public.stixor_fraud_features_distributed
🔍 Filter: Customer Account only


## 4. Define All Rule Queries

In [ ]:
# Dictionary to store all rule queries
rule_queries = {
    'rule_1': """
    WITH rule_1_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('day', toDate(mbar_registered_date_time), cutoff_date) < 30 
                    AND trx_amt > 10000 
                THEN 1 ELSE 0 
            END AS rule_1_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 1: New Account High Amount' AS rule_name,
        countIf(rule_1_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_1_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_1_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_1_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1_flag = 1), 0), 4) AS precision,
        round(countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2 * (countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1_flag = 1), 0)) * 
                  (countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)) /
              nullIf((countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_1_flag = 1), 0)) + 
                     (countIf(rule_1_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0)), 0), 4) AS f1_score,
        countIf(rule_1_flag = 1) AS total_flagged
    FROM rule_1_calc
    """,
    
    'rule_2': """
    WITH rule_2_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN dateDiff('hour', mbar_registered_date_time, trans_initiate_time) < 1 
                    AND trx_amt > 10000 
                THEN 1 ELSE 0 
            END AS rule_2_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 2: Very New Account' AS rule_name,
        countIf(rule_2_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_2_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_2_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_2_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_2_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_2_flag = 1), 0), 4) AS precision,
        round(countIf(rule_2_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_2_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_2_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_2_flag = 1) AS total_flagged
    FROM rule_2_calc
    """,
    
    'rule_3': """
    WITH rule_3_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN COUNT(*) OVER (
                    PARTITION BY ac_from 
                    ORDER BY toUnixTimestamp(trans_initiate_time) ASC 
                    RANGE BETWEEN 3600 PRECEDING AND CURRENT ROW
                ) > 10 
                THEN 1 ELSE 0 
            END AS rule_3_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 3: High Velocity' AS rule_name,
        countIf(rule_3_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_3_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_3_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_3_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_3_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_3_flag = 1), 0), 4) AS precision,
        round(countIf(rule_3_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_3_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_3_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_3_flag = 1) AS total_flagged
    FROM rule_3_calc
    """,
    
    'rule_4': """
    WITH rule_4_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN trx_amt >= 3 * MAX(trx_amt) OVER (
                    PARTITION BY ac_from 
                    ORDER BY toUnixTimestamp(trans_initiate_time)
                    RANGE BETWEEN 2592000 PRECEDING AND 1 PRECEDING
                ) 
                THEN 1 ELSE 0 
            END AS rule_4_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 4: 3x Max Amount' AS rule_name,
        countIf(rule_4_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_4_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_4_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_4_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_4_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_4_flag = 1), 0), 4) AS precision,
        round(countIf(rule_4_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_4_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_4_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_4_flag = 1) AS total_flagged
    FROM rule_4_calc
    """,
    
    'rule_5': """
    WITH rule_5_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN uniqExact(trx_channel) OVER (
                    PARTITION BY ac_from 
                    ORDER BY toUnixTimestamp(trans_initiate_time)
                    RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
                ) > 1 
                THEN 1 ELSE 0 
            END AS rule_5_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 5: Multiple Channels' AS rule_name,
        countIf(rule_5_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_5_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_5_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_5_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_5_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_5_flag = 1), 0), 4) AS precision,
        round(countIf(rule_5_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_5_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_5_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_5_flag = 1) AS total_flagged
    FROM rule_5_calc
    """,
    
    'rule_6': """
    WITH rule_6_calc AS (
        SELECT 
            trans_id,
            fraud_flag,
            CASE 
                WHEN toHour(trans_initiate_time) BETWEEN 1 AND 4 
                THEN 1 ELSE 0 
            END AS rule_6_flag
        FROM public.stixor_fraud_features_distributed
        WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
            AND mbar_account_type_name = 'Customer Account'
    )
    SELECT 
        'Rule 6: Off-Peak Hours' AS rule_name,
        countIf(rule_6_flag = 1 AND fraud_flag = 1) AS true_positives,
        countIf(rule_6_flag = 1 AND fraud_flag = 0) AS false_positives,
        countIf(rule_6_flag = 0 AND fraud_flag = 1) AS false_negatives,
        countIf(rule_6_flag = 0 AND fraud_flag = 0) AS true_negatives,
        round(countIf(rule_6_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(rule_6_flag = 1), 0), 4) AS precision,
        round(countIf(rule_6_flag = 1 AND fraud_flag = 1) * 100.0 / nullIf(countIf(fraud_flag = 1), 0), 4) AS recall,
        round(2.0 * countIf(rule_6_flag = 1 AND fraud_flag = 1) / nullIf(countIf(rule_6_flag = 1) + countIf(fraud_flag = 1), 0) * 100, 4) AS f1_score,
        countIf(rule_6_flag = 1) AS total_flagged
    FROM rule_6_calc
    """
}

print(f"✅ Defined {len(rule_queries)} fraud detection rules")
for rule_name in rule_queries.keys():
    print(f"   • {rule_name}")

## 5. Execute All Rules and Collect Results

In [ ]:
import time

# List to store results
all_results = []

print("🚀 Executing fraud detection rules in ClickHouse...\n")
print("="*80)

for rule_name, query_template in rule_queries.items():
    print(f"\n📊 Executing {rule_name.upper()}...")
    
    # Format query with date parameters
    query = query_template.format(start_date=START_DATE, end_date=END_DATE)
    
    try:
        start_time = time.time()
        
        # Execute query
        result = client.execute(query)
        
        execution_time = time.time() - start_time
        
        # Extract results
        if result:
            row = result[0]
            rule_data = {
                'rule_id': rule_name,
                'rule_name': row[0],
                'true_positives': row[1],
                'false_positives': row[2],
                'false_negatives': row[3],
                'true_negatives': row[4],
                'precision': row[5],
                'recall': row[6],
                'f1_score': row[7],
                'total_flagged': row[8],
                'execution_time_sec': round(execution_time, 2)
            }
            all_results.append(rule_data)
            
            print(f"   ✅ Completed in {execution_time:.2f} seconds")
            print(f"   • Precision: {row[5]}%")
            print(f"   • Recall: {row[6]}%")
            print(f"   • F1 Score: {row[7]}")
            print(f"   • Total Flagged: {row[8]:,}")
        else:
            print(f"   ⚠️ No results returned")
            
    except Exception as e:
        print(f"   ❌ Error: {str(e)}")
        continue

print("\n" + "="*80)
print(f"\n✅ Completed execution of {len(all_results)} rules")

## 6. Create DataFrame with All Results

In [ ]:
# Create DataFrame from results
rules_df = pd.DataFrame(all_results)

# Display results
print("📊 FRAUD RULES PERFORMANCE SUMMARY")
print("="*100)
print(rules_df.to_string(index=False))
print("="*100)

## 7. Analyze Results

In [ ]:
# Calculate FPR (False Positive Rate)
rules_df['fpr'] = (rules_df['false_positives'] / 
                   (rules_df['false_positives'] + rules_df['true_negatives']) * 100).round(4)

# Calculate Accuracy
rules_df['accuracy'] = ((rules_df['true_positives'] + rules_df['true_negatives']) / 
                        (rules_df['true_positives'] + rules_df['false_positives'] + 
                         rules_df['false_negatives'] + rules_df['true_negatives']) * 100).round(4)

print("\n📈 ADDITIONAL METRICS")
print("="*100)
print(rules_df[['rule_name', 'precision', 'recall', 'f1_score', 'fpr', 'accuracy']].to_string(index=False))
print("="*100)

## 8. Rank Rules by Performance

In [ ]:
# Top rules by Precision
print("\n🏆 TOP RULES BY PRECISION")
print("="*80)
top_precision = rules_df.nlargest(5, 'precision')[['rule_name', 'precision', 'recall', 'f1_score', 'total_flagged']]
print(top_precision.to_string(index=False))

# Top rules by Recall
print("\n\n🏆 TOP RULES BY RECALL")
print("="*80)
top_recall = rules_df.nlargest(5, 'recall')[['rule_name', 'recall', 'precision', 'f1_score', 'total_flagged']]
print(top_recall.to_string(index=False))

# Top rules by F1 Score
print("\n\n🏆 TOP RULES BY F1 SCORE")
print("="*80)
top_f1 = rules_df.nlargest(5, 'f1_score')[['rule_name', 'f1_score', 'precision', 'recall', 'total_flagged']]
print(top_f1.to_string(index=False))

# Lowest FPR
print("\n\n🎯 LOWEST FALSE POSITIVE RATE")
print("="*80)
low_fpr = rules_df.nsmallest(5, 'fpr')[['rule_name', 'fpr', 'precision', 'recall', 'total_flagged']]
print(low_fpr.to_string(index=False))

## 9. Summary Statistics

In [ ]:
print("\n📊 SUMMARY STATISTICS")
print("="*80)

summary = {
    'Metric': ['Precision (%)', 'Recall (%)', 'F1 Score', 'FPR (%)', 'Accuracy (%)'],
    'Mean': [
        rules_df['precision'].mean(),
        rules_df['recall'].mean(),
        rules_df['f1_score'].mean(),
        rules_df['fpr'].mean(),
        rules_df['accuracy'].mean()
    ],
    'Median': [
        rules_df['precision'].median(),
        rules_df['recall'].median(),
        rules_df['f1_score'].median(),
        rules_df['fpr'].median(),
        rules_df['accuracy'].median()
    ],
    'Std Dev': [
        rules_df['precision'].std(),
        rules_df['recall'].std(),
        rules_df['f1_score'].std(),
        rules_df['fpr'].std(),
        rules_df['accuracy'].std()
    ],
    'Min': [
        rules_df['precision'].min(),
        rules_df['recall'].min(),
        rules_df['f1_score'].min(),
        rules_df['fpr'].min(),
        rules_df['accuracy'].min()
    ],
    'Max': [
        rules_df['precision'].max(),
        rules_df['recall'].max(),
        rules_df['f1_score'].max(),
        rules_df['fpr'].max(),
        rules_df['accuracy'].max()
    ]
}

summary_df = pd.DataFrame(summary)
summary_df = summary_df.round(4)
print(summary_df.to_string(index=False))
print("="*80)

## 10. Save Results to CSV

In [ ]:
# Save to CSV
output_file = '/root/research-dir/dev/jazzcash-fraud-detection/data/fraud_rules_clickhouse_results.csv'
rules_df.to_csv(output_file, index=False)

print(f"\n💾 Results saved to: {output_file}")
print(f"\n📊 DataFrame shape: {rules_df.shape}")
print(f"   • Rows: {rules_df.shape[0]}")
print(f"   • Columns: {rules_df.shape[1]}")

## 11. Categorize Rules by Performance

In [ ]:
def categorize_rule(row):
    """Categorize rule based on precision and FPR"""
    if row['precision'] >= 80 and row['fpr'] < 5:
        return 'EXCELLENT'
    elif row['precision'] >= 60 and row['fpr'] < 10:
        return 'GOOD'
    elif row['precision'] >= 40 and row['fpr'] < 20:
        return 'MODERATE'
    else:
        return 'POOR'

rules_df['category'] = rules_df.apply(categorize_rule, axis=1)

print("\n🏷️  RULE CATEGORIZATION")
print("="*80)
print(rules_df[['rule_name', 'precision', 'fpr', 'f1_score', 'category']].to_string(index=False))
print("\n")

# Count by category
category_counts = rules_df['category'].value_counts()
print("Category Distribution:")
for category, count in category_counts.items():
    print(f"   • {category}: {count} rules")

print("="*80)

## 12. Close ClickHouse Connection

In [ ]:
# Disconnect from ClickHouse
client.disconnect()

print("✅ ClickHouse connection closed")
print("\n🎉 Analysis Complete!")